In [4]:
import jieba
from gensim import corpora, models
from gensim.models import Word2Vec, KeyedVectors
import warnings
warnings.filterwarnings('ignore')

## 二、Gensim工作流

In [13]:
raw_headlines = [
    "央行降息，刺激股市反弹",
    "球队赢得总决赛冠军，球员表现出色"
]
tokenized_headlines=[jieba.lcut(doc)for doc in raw_headlines]
print(f"分词后语料: {tokenized_headlines}")
dictionary=corpora.Dictionary(tokenized_headlines)
print(f"词典: {dictionary.token2id}")
corpus_bow=[dictionary.doc2bow(doc)for doc in tokenized_headlines]
print(f"BoW语料库: {corpus_bow}")

分词后语料: [['央行', '降息', '，', '刺激', '股市', '反弹'], ['球队', '赢得', '总决赛', '冠军', '，', '球员', '表现出色']]
词典: {'刺激': 0, '反弹': 1, '央行': 2, '股市': 3, '降息': 4, '，': 5, '冠军': 6, '总决赛': 7, '球员': 8, '球队': 9, '表现出色': 10, '赢得': 11}
BoW语料库: [[(0, 1), (1, 1), (2, 1), (3, 1), (4, 1), (5, 1)], [(5, 1), (6, 1), (7, 1), (8, 1), (9, 1), (10, 1), (11, 1)]]


## 三、TF-IDF与关键词权重

In [17]:
headlines = [
    "央行降息，刺激股市反弹",
    "球队赢得总决赛冠军，球员表现出色",
    "国家队公布最新一期足球集训名单",
    "A股市场持续震荡，投资者需谨慎",
    "篮球巨星刷新历史得分记录",
    "理财产品收益率创下新高"
]
tokenized_headlines = [jieba.lcut(title) for title in headlines]

# 2. 创建词典和 BoW 语料库
dictionary = corpora.Dictionary(tokenized_headlines)
corpus_bow = [dictionary.doc2bow(doc) for doc in tokenized_headlines]

# 3. 训练 TF-IDF 模型
tfidf_model = models.TfidfModel(corpus_bow)

# 4. 将BoW语料库转换为 TF-IDF 向量表示
corpus_tfidf = tfidf_model[corpus_bow]
def tfidf_with_words(tfidf_vec,id2word):
    pairs=[(id2word[token_id],weight)for token_id ,weight in tfidf_vec]
    return sorted(pairs,key=lambda x:x[1],reverse=True)
first_tfidf = list(corpus_tfidf)[0]
print("第一篇标题的 TF-IDF 向量:")
print(first_tfidf)
print("第一篇标题的 TF-IDF 向量(带词语):")
print(tfidf_with_words(first_tfidf, dictionary))
new_headline = "股市大涨，牛市来了"
new_headline_bow = dictionary.doc2bow(jieba.lcut(new_headline))
new_headline_tfidf = tfidf_model[new_headline_bow]
print("\n新标题的 TF-IDF 向量:")
print(new_headline_tfidf)

第一篇标题的 TF-IDF 向量:
[(0, np.float64(0.44066740566370055)), (1, np.float64(0.44066740566370055)), (2, np.float64(0.44066740566370055)), (3, np.float64(0.44066740566370055)), (4, np.float64(0.44066740566370055)), (5, np.float64(0.1704734229377651))]
第一篇标题的 TF-IDF 向量(带词语):
[('刺激', np.float64(0.44066740566370055)), ('反弹', np.float64(0.44066740566370055)), ('央行', np.float64(0.44066740566370055)), ('股市', np.float64(0.44066740566370055)), ('降息', np.float64(0.44066740566370055)), ('，', np.float64(0.1704734229377651))]

新标题的 TF-IDF 向量:
[(3, np.float64(0.9326446771245245)), (5, np.float64(0.360796211497975))]


## 四、LDA与文档主题挖掘

In [8]:
# 1. 准备语料
headlines = [
    "央行降息，刺激股市反弹",
    "球队赢得总决赛冠军，球员表现出色",
    "国家队公布最新一期足球集训名单",
    "A股市场持续震荡，投资者需谨慎",
    "篮球巨星刷新历史得分记录",
    "理财产品收益率创下新高"
]
tokenized_headlines = [jieba.lcut(title) for title in headlines]

# 2. 创建词典和 BoW 语料库
dictionary = corpora.Dictionary(tokenized_headlines)
corpus_bow = [dictionary.doc2bow(doc) for doc in tokenized_headlines]

# 3. 训练 LDA 模型 (假设需要发现 2 个主题)
lda_model = models.LdaModel(corpus=corpus_bow, id2word=dictionary, num_topics=2, random_state=100)

# 4. 查看模型发现的主题
print("模型发现的 2 个主题及其关键词:")
for topic in lda_model.print_topics():
    print(topic)


模型发现的 2 个主题及其关键词:
(0, '0.045*"，" + 0.040*"公布" + 0.039*"一期" + 0.039*"名单" + 0.039*"足球" + 0.039*"最新" + 0.038*"集训" + 0.038*"国家队" + 0.037*"A股" + 0.037*"市场"')
(1, '0.066*"，" + 0.039*"篮球" + 0.039*"刷新" + 0.039*"历史" + 0.039*"记录" + 0.038*"得分" + 0.038*"巨星" + 0.037*"刺激" + 0.036*"降息" + 0.036*"反弹"')


In [9]:
# 5. 推断新文档的主题分布
new_headline = "巨星詹姆斯获得常规赛MVP"
new_headline_bow = dictionary.doc2bow(jieba.lcut(new_headline))
topic_distribution = lda_model[new_headline_bow]
print(f"\n新标题 '{new_headline}' 的主题分布:")
print(topic_distribution)


新标题 '巨星詹姆斯获得常规赛MVP' 的主题分布:
[(0, np.float32(0.27243596)), (1, np.float32(0.72756404))]


## 五、Word2Vec模型实战

### 5.1 模型训练与核心参数

In [10]:
# 1. 准备语料 
headlines = [
    # 财经
    "央行降息，刺激股市反弹",
    "A股市场持续震荡，投资者需谨慎",
    "理财产品收益率创下新高",
    "证监会发布新规，规范市场交易",
    "创业板指数上涨，科技股领涨大盘",
    "房价调控政策出台，房地产市场降温",
    "全球股市动荡，影响资本市场信心",
    "分析师认为，当前股市风险与机遇并存，市场情绪复杂",

    # 体育
    "球队赢得总决赛冠军，球员表现出色",
    "国家队公布最新一期足球集训名单",
    "篮球巨星刷新历史得分记录",
    "奥运会开幕，中国代表团旗手确定",
    "马拉松比赛圆满结束，选手创造佳绩",
    "电子竞技联赛吸引大量年轻观众",
    "这支球队的每位球员都表现出色",
    "球员转会市场活跃，多支球队积极引援"
]

tokenized_headlines = [jieba.lcut(title) for title in headlines]

# 2. 训练 Word2Vec 模型
model = Word2Vec(tokenized_headlines, vector_size=50, window=3, min_count=1, sg=1)

print(f"模型训练完成！")
print(f"词汇表大小: {len(model.wv)}")
print(f"词向量维度: {model.wv.vector_size}")

# 训练完成后，所有词向量都存储在 model.wv 对象中
# model.wv 是一个 KeyedVectors 实例

模型训练完成！
词汇表大小: 93
词向量维度: 50


### 5.2 使用词向量


In [ ]:

similar_to_market = model.wv.most_similar('股市')
print(f"与 '股市' 最相似的词: {similar_to_market}")


similarity = model.wv.similarity('球队', '球员')
print(f"\n'球队' 和 '球员' 的相似度: {similarity:.4f}")

market_vector = model.wv['市场']
print(f"\n'市场' 的向量维度: {market_vector.shape}")

与 '股市' 最相似的词: [('赢得', 0.3200092017650604), ('的', 0.30753466486930847), ('出台', 0.28610458970069885), ('领涨', 0.2767459452152252), ('大量', 0.2701166272163391), ('房价', 0.23572921752929688), ('指数', 0.23326583206653595), ('交易', 0.2314068228006363), ('活跃', 0.2246733158826828), ('反弹', 0.2168722301721573)]

'球队' 和 '球员' 的相似度: -0.1547

'市场' 的向量维度: (50,)


### 5.3 模型的持久化


In [12]:
# 保存词向量到文件
model.wv.save("news_vectors.kv")

# 从文件加载词向量
loaded_wv = KeyedVectors.load("news_vectors.kv")

# 加载后可以执行同样的操作
print(f"\n加载后，'球队' 和 '球员' 的相似度: {loaded_wv.similarity('球队', '球员'):.4f}")


加载后，'球队' 和 '球员' 的相似度: -0.1547
